# Telecom Customer Churn :


- Compare Logistic Regression, Random Forest and XGBoost.
- Evaluate accuracy **and** churn-focused metrics.
- Tune XGBoost using F1 rather than accuracy.
- Analyze the probability threshold for churn.
- Add feature importance and business-oriented churn-risk analysis.

In [ ]:
# Import important libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_curve, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42


In [ ]:
# Load the dataset
df=pd.read_csv(r'Telco-Customer-Churn.csv')
df

## 1. Prepare a clean modeling dataframe

We remove the customer ID because it is an identifier, not a meaningful predictive feature.

`TotalCharges` is converted to numeric. Blank values become `NaN` and are handled **inside the preprocessing pipeline**, which prevents test-set information from leaking into training.


In [ ]:
# Work on a copy so the original dataframe remains unchanged
data = df.copy()

# Remove identifier
if "customerID" in data.columns:
    data = data.drop(columns=["customerID"])

# Convert TotalCharges to numeric
if "TotalCharges" in data.columns:
    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

# Target encoding
data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})

print("Shape:", data.shape)
print("\nMissing values:")
print(data.isna().sum().sort_values(ascending=False).head(10))
print("\nTarget distribution:")
print(data["Churn"].value_counts(normalize=True).rename({0:"No", 1:"Yes"}))


## 2. Business-oriented EDA

The following plots focus on variables that can help explain *why* customers churn, rather than only showing generic distributions.


In [ ]:
# Helper function: churn rate by category
def churn_rate_by_category(data, column):
    result = data.groupby(column, dropna=False)["Churn"].mean().sort_values(ascending=False)
    return (result * 100).round(2)

eda_cols = [
    c for c in [
        "Contract", "InternetService", "PaymentMethod",
        "TechSupport", "OnlineSecurity", "OnlineBackup"
    ] if c in data.columns
]

for col in eda_cols:
    print(f"\n{col} - churn rate (%)")
    print(churn_rate_by_category(data, col))

    churn_rates = churn_rate_by_category(data, col)
    plt.figure(figsize=(8, 4))
    churn_rates.plot(kind="bar")
    plt.ylabel("Churn rate (%)")
    plt.title(f"Churn Rate by {col}")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# Tenure and MonthlyCharges by churn
fig, ax = plt.subplots(figsize=(8, 4))
data.boxplot(column="tenure", by="Churn", ax=ax)
ax.set_title("Tenure by Churn")
ax.set_xlabel("Churn (0 = No, 1 = Yes)")
ax.set_ylabel("Tenure (months)")
plt.suptitle("")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
data.boxplot(column="MonthlyCharges", by="Churn", ax=ax)
ax.set_title("Monthly Charges by Churn")
ax.set_xlabel("Churn (0 = No, 1 = Yes)")
ax.set_ylabel("Monthly Charges")
plt.suptitle("")
plt.tight_layout()
plt.show()


### EDA interpretation

For a telecom retention strategy, particular observations are:

1. 'Month to month' contract type has the highest churn rate.
2. Low tenure is the strongest signal for churn risk.
3. High monthly charges are associated with higher churn.
4. Fibre optic internet-service group has the highest churn.
5. Provision of support/security services reduce the churn.
6. Electronic check payment method shows highest churn rate, whereas it is almost similar for other payment methods.


## 3. Train/test split BEFORE fitting preprocessing

This is the key leakage fix.

The imputer and encoders are now fitted only on the training data through the pipeline.


In [ ]:
X = data.drop(columns=["Churn"])
y = data["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


In [ ]:
# Robust preprocessing
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
])


## 4. Baseline model: Logistic Regression

Logistic Regression provides an interpretable baseline. Tree models can then be judged against it rather than evaluated in isolation.


In [ ]:
logreg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

logreg_pipeline.fit(X_train, y_train)

def evaluate_model(model, X_test, y_test, name):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, prob)
    }
    print(classification_report(y_test, pred, digits=3))
    return metrics

results = []
results.append(evaluate_model(logreg_pipeline, X_test, y_test, "Logistic Regression"))


## 5. Random Forest comparison


In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
results.append(evaluate_model(rf_pipeline, X_test, y_test, "Random Forest"))


## 6. XGBoost model

We use `scale_pos_weight` based on the training class distribution to give the minority churn class more importance.


In [ ]:
negative, positive = np.bincount(y_train)
scale_pos_weight = negative / positive

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight
    ))
])

xgb_pipeline.fit(X_train, y_train)
results.append(evaluate_model(xgb_pipeline, X_test, y_test, "XGBoost"))


In [ ]:
comparison = pd.DataFrame(results).sort_values("F1", ascending=False)
comparison


## 7. Hyperparameter tuning

For churn prediction, optimizing only for accuracy can favor the majority class. Here we optimize **F1**, which balances precision and recall for the churn class.


In [ ]:
param_distributions = {
    "model__n_estimators": np.arange(150, 601, 50),
    "model__max_depth": np.arange(3, 9),
    "model__learning_rate": np.linspace(0.01, 0.20, 20),
    "model__subsample": np.linspace(0.70, 1.00, 7),
    "model__colsample_bytree": np.linspace(0.70, 1.00, 7),
    "model__min_child_weight": np.arange(1, 8),
    "model__gamma": np.linspace(0, 0.4, 9)
}

search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="f1",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

best_xgb = search.best_estimator_

print("Best CV F1:", search.best_score_)
print("Best parameters:")
print(search.best_params_)


In [ ]:
tuned_metrics = evaluate_model(best_xgb, X_test, y_test, "Tuned XGBoost")
comparison = pd.concat([comparison, pd.DataFrame([tuned_metrics])], ignore_index=True)
comparison.sort_values("F1", ascending=False)


## 8. Confusion matrix for the final model


In [ ]:
y_pred = best_xgb.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print(cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Churn", "Churn"]
).plot()
plt.title("Tuned XGBoost - Confusion Matrix")
plt.tight_layout()
plt.show()


## 9. Threshold tuning

`predict()` uses a default probability threshold. For churn prevention, a lower threshold may be worthwhile because missing a true churner can be costly.

The code below compares several thresholds. Select a threshold based on the business cost of false positives versus false negatives.
Lowering the threshold (e.g., to 0.30): Flags more customers as churn risks. This increases Recall (catches more churners) at the cost of lower Precision (more false alarms).

In [ ]:
y_prob = best_xgb.predict_proba(X_test)[:, 1]

threshold_rows = []
for threshold in np.arange(0.20, 0.71, 0.05):
    pred_t = (y_prob >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": round(threshold, 2),
        "Precision": precision_score(y_test, pred_t),
        "Recall": recall_score(y_test, pred_t),
        "F1": f1_score(y_test, pred_t),
        "Accuracy": accuracy_score(y_test, pred_t)
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(threshold_df["Threshold"], threshold_df["Precision"], marker="o", label="Precision")
plt.plot(threshold_df["Threshold"], threshold_df["Recall"], marker="o", label="Recall")
plt.plot(threshold_df["Threshold"], threshold_df["F1"], marker="o", label="F1")
plt.xlabel("Probability Threshold")
plt.ylabel("Score")
plt.title("Precision / Recall / F1 vs Churn Threshold")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 10. ROC and Precision-Recall curves


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
precision, recall, _ = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve (AUC = {roc_auc_score(y_test, y_prob):.3f})")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.tight_layout()
plt.show()


## 11. Feature importance

The following section provides a simple XGBoost feature-importance view.


In [ ]:
# Get transformed feature names
pre = best_xgb.named_steps["preprocessor"]
model = best_xgb.named_steps["model"]

feature_names = pre.get_feature_names_out()
importance = pd.Series(model.feature_importances_, index=feature_names)
importance = importance.sort_values(ascending=False)

top_features = importance.head(20).sort_values()

plt.figure(figsize=(9, 7))
top_features.plot(kind="barh")
plt.xlabel("Importance")
plt.title("Top 20 XGBoost Feature Importances")
plt.tight_layout()
plt.show()

top_features.sort_values(ascending=False).head(20)


## 12. Customer-level churn risk

A business application should output a probability, not just 0/1.


In [ ]:
customer_risk = X_test.copy()
customer_risk["ActualChurn"] = y_test.values
customer_risk["ChurnProbability"] = y_prob

customer_risk["RiskLevel"] = pd.cut(
    customer_risk["ChurnProbability"],
    bins=[-0.01, 0.30, 0.60, 0.80, 1.00],
    labels=["Low", "Medium", "High", "Very High"]
)

customer_risk.sort_values("ChurnProbability", ascending=False).head(20)


## Print the customer count according to Risk Level

In [ ]:
customer_risk["RiskLevel"].value_counts()

## 13. Final business interpretation

Use the model to prioritize retention efforts:

- **Very High risk:** immediate retention outreach.
- **High risk:** targeted offer or service intervention.
- **Medium risk:** monitor and communicate proactively.
- **Low risk:** normal customer management.

The model predicts *risk*, not the reason a customer will churn and not a causal effect of any particular service.
